**Bangladesh**

3 Datasets: Meeresspiegeldaten,  Flutrisiko-Shapefile und Niederschlag (NASA POWER)

1) Flutrisiko-Shapefile laden und als GeoJSON exportieren

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
%matplotlib inlie

floods = gpd.read_file('/Users/aniellepeterhans/Lede/Data_Project2/docs/bgd_nhr_floods_barc')
floods_clean = floods[['FLOODCAT', 'FLOODCAT_L', 'DISTNAME', 'geometry']].copy()

# Geometrie vereinfachen, um die Dateigrösse zu reduzieren (10.9 MB -> ca. 5.6 MB)
floods_clean['geometry'] = floods_clean['geometry'].simplify(0.001, preserve_topology=True)

floods_clean.to_file('flood_risk.geojson', driver='GeoJSON')

DataSourceError: /Users/aniellepeterhans/Lede/Data_Project2/docs/bgd_nhr_floods_barc: No such file or directory

2) Meeresspiegel-Daten parsen (Format: Jahr; mm; Flag; fehlende_Tage, -99999 = fehlender Wert):

In [34]:
import pandas as pd

bob = pd.read_csv('/Users/aniellepeterhans/Lede/Data_Project2/Bangladesh/slr_sla_bob_free_all_66.csv', comment='#')

mission_cols = ['TOPEX/Poseidon', 'Jason-1', 'Jason-2', 'Jason-3', 'Sentinel-6MF']
bob['sla_mm'] = bob[mission_cols].bfill(axis=1).iloc[:, 0]  # eine Mission pro Zeile -> zusammenführen
bob['year_int'] = bob['year'].astype(int)

sea_level = bob.groupby('year_int')['sla_mm'].mean().reset_index()
sea_level.columns = ['year', 'sla_mm']
print(sea_level)

    year      sla_mm
0   1992    4.540000
1   1993   -9.247353
2   1994  -20.051176
3   1995    2.290938
4   1996    3.833636
5   1997  -25.894242
6   1998   -9.344848
7   1999   -5.736667
8   2000   12.666176
9   2001    2.179167
10  2002   -7.341045
11  2003    2.084384
12  2004    5.237746
13  2005   17.960769
14  2006   -0.087500
15  2007   -3.881622
16  2008   39.060000
17  2009   26.051250
18  2010   52.170946
19  2011   48.824384
20  2012   45.726957
21  2013   61.916607
22  2014   44.282222
23  2015   66.018108
24  2016   97.112923
25  2017   62.230784
26  2018   48.582703
27  2019   46.292703
28  2020   86.901081
29  2021  129.564444
30  2022  106.641268
31  2023   53.022703
32  2024   88.315000
33  2025  160.430000


In [36]:
# 1992 und 2025 sind unvollständige Jahre, verzerrt deshlab raus. 
sea_level = sea_level[(sea_level['year'] >= 1993) & (sea_level['year'] <= 2024)]

In [41]:
import json

sealevel_json = [{"year": int(r['year']), "sla_mm": round(r['sla_mm'], 1)} for _, r in sea_level.iterrows()]
with open('sea_level_bay_of_bengal.json', 'w') as f:
    json.dump(sealevel_json, f)

3) Niederschlag aus NASA-POWER-Antwort

In [43]:
import requests
url = "https://power.larc.nasa.gov/api/temporal/monthly/point"
params = {
    "parameters": "PRECTOTCORR",
    "community": "AG",
    "longitude": 90.4125, "latitude": 23.8103,  # Dhaka
    "start": 1990, "end": 2023, "format": "JSON"
}
data = requests.get(url, params=params).json()

In [44]:
data.keys()

dict_keys(['type', 'geometry', 'properties', 'header', 'messages', 'parameters', 'times'])

In [45]:
import calendar

monthly = data['properties']['parameter']['PRECTOTCORR']

rows = []
for k, v in monthly.items():
    year, month = int(k[:4]), int(k[4:])
    if month > 12:
        continue  # der 13. Eintrag pro Jahr ist der Jahresdurchschnitt, kein Monat
    days = calendar.monthrange(year, month)[1]
    rows.append((year, v * days))  # mm/Tag -> mm für den Monat

precip = pd.DataFrame(rows, columns=['year', 'mm'])
precip_annual = precip.groupby('year')['mm'].sum().reset_index()
print(precip_annual)

    year       mm
0   1990  1099.57
1   1991  1184.62
2   1992   725.13
3   1993  1256.43
4   1994   788.51
5   1995  1164.18
6   1996  1030.13
7   1997  1606.79
8   1998  1920.74
9   1999  1631.48
10  2000  1978.81
11  2001  1887.09
12  2002  1890.85
13  2003  1744.66
14  2004  2164.52
15  2005  2038.64
16  2006  1765.92
17  2007  2561.80
18  2008  1598.15
19  2009  1802.80
20  2010  1779.31
21  2011  1620.89
22  2012  1506.60
23  2013  1663.65
24  2014  1704.33
25  2015  2143.35
26  2016  2546.93
27  2017  4630.59
28  2018  2705.77
29  2019  3002.42
30  2020  2867.28
31  2021  2336.75
32  2022  2262.49
33  2023  2553.02


In [ ]:
# 2017: 4630.59 (!) da gab es tatsächlich eine der schwersten Flutkatastrophen in Bangladesch. Im Text aufnehmen. 

In [3]:
import json

#Niederschlag als JSON (gleiche Struktur wie permafrost_timeseries.json)
precip_json = [{"year": int(r['year']), "precip_mm": round(r['mm'], 1)} for _, r in precip_annual.iterrows()]
with open('precipitation_dhaka.json', 'w') as f:
    json.dump(precip_json, f)

print("Fertig:", len(precip_json), "Niederschlagsjahre,", len(sealevel_json), "Meeresspiegeljahre")

NameError: name 'precip_annual' is not defined